# Tutorial 3: Interpreting DASH Outputs

DASH produces three types of diagnostic outputs. This tutorial answers three clinical questions using Breast Cancer:

1. **Which features are trustworthy?** — The IS plot shows at a glance which features have stable, individually-meaningful attributions vs. which are contested collinear cluster members
2. **Which features should be reported as a group?** — Quadrant II features (Collinear Cluster) should have their importance aggregated before reporting
3. **For a specific patient, which features are contested?** — Local disagreement maps show per-observation attribution uncertainty

**Prerequisites**: [Tutorial 2](tutorial_02_dash_walkthrough.ipynb) explained how each output is computed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from dash_shap import DASHPipeline

TUTORIAL_MODE = True
M = 30 if TUTORIAL_MODE else 200
K = 10 if TUTORIAL_MODE else 30

bc = load_breast_cancer()
X, y = bc.data, bc.target
feature_names = list(bc.feature_names)

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_temp, X_explain, y_temp, _ = train_test_split(X_temp, y_temp, test_size=0.12, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.18, random_state=42)

print(f"Split: X_train={len(X_train)}, X_val={len(X_val)}, X_explain={len(X_explain)}, X_test={len(X_test)}")
print(f"Note: Paper uses M=200, K=30. Tutorial uses M={M}, K={K} for speed.")

pipe = DASHPipeline(M=M, K=K, epsilon=0.05, epsilon_mode="relative",
                    task="binary", seed=42, verbose=False)
pipe.fit(X_train, y_train, X_val, y_val, X_ref=X_explain, feature_names=feature_names)
print(f"\nFit complete. K={len(pipe.selected_indices_)} models selected.")

## The IS Plot — How to Read It

The Importance-Stability (IS) plot places every feature at the point:
- **x-axis**: consensus importance = mean |SHAP| across the K models (how much this feature contributes to predictions on average)
- **y-axis**: FSI = standard deviation of SHAP attributions across models / mean absolute SHAP (how much the K models *disagree* about this feature)

Dividing lines are at the **median importance** (vertical) and the **median FSI among high-importance features** (horizontal). This makes the quadrant boundaries data-adaptive — they scale to the dataset.

**Code's quadrant convention** (from `diagnostics.py`):

| Quadrant | Importance | FSI | Interpretation |
|---|---|---|---|
| I: Robust Drivers | High | **Low** | Trustworthy — important and stable; report individually |
| II: Collinear Cluster | High | **High** | Interpret as a group — important but contested by correlated features |
| III: Confirmed Unimportant | Low | Low | Reliably irrelevant — safe to omit |
| IV: Fragile Interactions | Low | High | Unstable and weak — investigate or de-emphasize |

For Breast Cancer, we expect the radius cluster (mean radius, mean perimeter, mean area, worst radius, worst perimeter, worst area) to appear in Quadrant II — high consensus importance (tumor size matters for malignancy) but high FSI (the K models disagree about which size measurement to credit).

In [ ]:
fig = pipe.plot_importance_stability(title=f"Breast Cancer — IS Plot (M={M}, K={K})")
plt.show()
print("IS Plot: feature_names_ were stored during fit(); no need to pass them again.")
print("Quadrant assignment uses median thresholds (data-adaptive).")

## Guided Tour of the IS Plot

With `M=30, K=10` (tutorial mode), the IS plot gives indicative positions. At the paper's `M=200, K=30`, the positions are more stable but the qualitative pattern is the same.

**What to look for:**

- **Quadrant II cluster** (top-right region): The radius/size features — mean radius, mean perimeter, mean area, worst radius, worst perimeter, worst area — likely appear here together. Their high consensus importance means tumor size matters, but their high FSI means the K models disagree about *which* size measurement to credit. This is first-mover bias in action: within the tumor-size cluster, attribution is unstable.

- **Quadrant I features** (bottom-right): Features that appear individually informative beyond what their correlated partners can explain. "Worst concave points" may appear here if the concavity cluster has a clearer dominant member than the radius cluster.

- **Quadrant III** (bottom-left): Features consistently unimportant across all K models — confidently omit from reports.

Note: with M=30, K=10, some features may shift quadrants compared to the paper (M=200, K=30). The cluster structure should still be visible.

In [ ]:
fsi = pipe.get_fsi()
print(fsi.summary(top_k=15))
print("\nFSI thresholds (rough guide):")
print("  FSI < 0.3   : stable — report individually")
print("  FSI 0.3–0.7 : moderate — may be collinear, check IS plot")
print("  FSI > 0.7   : interpret with caution — likely collinear cluster member")

In [ ]:
labels = fsi.get_quadrant_labels()

print("Quadrant assignments for all 30 features:")
print(f"{'Feature':<30} {'Quadrant':<30} {'Action'}")
print("-" * 90)

action_map = {
    "I: Robust Drivers": "Report individually",
    "II: Collinear Cluster": "Report as group total",
    "III: Confirmed Unimportant": "Omit from report",
    "IV: Fragile Interactions": "Investigate further",
}
order = np.argsort(pipe.global_importance_)[::-1]
for j in order:
    print(f"{feature_names[j]:<30} {labels[j]:<30} {action_map.get(labels[j], '')}")

## Action Guide

| Quadrant | Interpretation | Recommended Action | Example (Breast Cancer) |
|---|---|---|---|
| I: Robust Drivers | High importance, low FSI | Report individually | "worst concave points is a robust driver of malignancy prediction" |
| II: Collinear Cluster | High importance, high FSI | Report the group | "mean radius, mean perimeter, worst radius form a size cluster — combined importance: X" |
| III: Confirmed Unimportant | Low importance, low FSI | Omit from report | "mean fractal dimension is consistently unimportant across all K models" |
| IV: Fragile Interactions | Low importance, high FSI | Investigate further | "mean symmetry shows unstable attribution — may reflect interaction with other features" |

**Key principle**: FSI tells you when a feature's attribution is contested. For contested (high-FSI) features with high importance, the question is not "which feature is most important" but "how much does the tumor-size cluster collectively contribute?"

## Group-Level Reporting

When features are in Quadrant II (Collinear Cluster), the interpretable quantity is the **group's combined importance**, not any individual member. Reporting "worst radius: 0.08" is misleading when the K models disagree about whether the credit belongs to worst radius or worst perimeter.

For the Breast Cancer radius cluster, we sum the consensus importance across all cluster members.

In [ ]:
RADIUS_CLUSTER = ["mean radius", "mean perimeter", "mean area",
                  "worst radius", "worst perimeter", "worst area"]
CONCAVITY_CLUSTER = ["mean concavity", "mean concave points",
                     "worst concavity", "worst concave points"]

radius_mask = np.array([f in RADIUS_CLUSTER for f in feature_names])
concavity_mask = np.array([f in CONCAVITY_CLUSTER for f in feature_names])

radius_importance = np.sum(pipe.global_importance_[radius_mask])
concavity_importance = np.sum(pipe.global_importance_[concavity_mask])
total_importance = np.sum(pipe.global_importance_)

print("Group-level importance (Breast Cancer):")
print(f"\n  Radius/size cluster:   {radius_importance:.4f}  ({radius_importance/total_importance:.1%} of total)")
print(f"  Members: {', '.join(RADIUS_CLUSTER)}")
print(f"\n  Concavity cluster:     {concavity_importance:.4f}  ({concavity_importance/total_importance:.1%} of total)")
print(f"  Members: {', '.join(CONCAVITY_CLUSTER)}")
print(f"\n  Total importance (all features): {total_importance:.4f}")
print(f"\nThe radius cluster's combined importance is more informative than")
print(f"the apparent importance of any individual member.")

## Local Disagreement Maps

The IS plot and FSI summarize feature stability *globally* — averaged over all reference observations. But individual patients may have very different uncertainty profiles.

**Local disagreement maps** show, for a single observation, the consensus SHAP value (bar center) and the cross-model standard deviation (error bars). Wide error bars mean the K models disagree about how to attribute this particular patient's risk.

Features with wide error bars are the same collinear cluster members we saw in the IS plot — but now we can see whether this specific patient's prediction is driven by a contested feature.

In [ ]:
# Find most contested observation: highest mean variance across features
mean_var_per_obs = np.mean(pipe.variance_matrix_, axis=1)
max_var_idx = int(np.argmax(mean_var_per_obs))
print(f"Most contested patient: observation {max_var_idx}")
print(f"  Mean SHAP variance: {mean_var_per_obs[max_var_idx]:.4f}")
print(f"  (Average across all {len(mean_var_per_obs)} patients: {np.mean(mean_var_per_obs):.4f})")

In [ ]:
from dash_shap.core.diagnostics import local_disagreement_map

fig = local_disagreement_map(
    pipe.all_shap_matrices_,
    max_var_idx,
    feature_names=feature_names,
    top_k=12,
    title=f"Patient {max_var_idx} — Local Disagreement Map (Most Contested)"
)
plt.show()
print("Wide error bars = the K models disagree about this feature's contribution for this patient.")
print("The contested features are typically the radius/concavity cluster members.")

## Interpreting the Local Disagreement Map

For the most contested patient (observation above):

- **Narrow error bars**: the K models agree on this feature's contribution — trustworthy for this patient
- **Wide error bars**: the K models disagree — likely a cluster member where the first-mover effect is particularly strong for this patient

The **consensus bar center** is your best estimate of this feature's contribution. The error bar represents *model uncertainty* about the attribution, not prediction uncertainty. Even if the K models disagree about whether "worst radius" or "worst perimeter" drove this patient's high risk score, they likely agree that the *size cluster collectively* was important.

Wide error bars do not mean the explanation is useless — they tell you that you should report the cluster total, not the individual feature.

In [ ]:
# Find least contested observation for contrast
min_var_idx = int(np.argmin(mean_var_per_obs))
print(f"Least contested patient: observation {min_var_idx}")
print(f"  Mean SHAP variance: {mean_var_per_obs[min_var_idx]:.6f}")

fig = local_disagreement_map(
    pipe.all_shap_matrices_,
    min_var_idx,
    feature_names=feature_names,
    top_k=12,
    title=f"Patient {min_var_idx} — Local Disagreement Map (Least Contested)"
)
plt.show()
print("Narrow error bars = all K models agree on this patient's attributions.")

## The "Boring" IS Plot — What Healthy Looks Like

The IS plot for a dataset with no collinearity is instructive: all features should have low FSI (near zero), and most should land in Quadrant I (Robust Drivers) or Quadrant III (Confirmed Unimportant). No Quadrant II cluster.

We generate a quick synthetic dataset with ρ=0.0 (independent features) to show this baseline.

In [ ]:
from dash_shap.experiments.synthetic import generate_synthetic_linear

(X_tr0, y_tr0, X_v0, y_v0, X_exp0, _, X_ts0, _, groups0, _, _) = generate_synthetic_linear(
    N=800, P=10, group_size=5, rho=0.0, seed=42
)

pipe0 = DASHPipeline(M=20, K=5, epsilon=0.10, task="regression", seed=42, verbose=False)
pipe0.fit(X_tr0, y_tr0, X_v0, y_v0, X_ref=X_exp0)

fig0 = pipe0.plot_importance_stability(title="IS Plot — No Collinearity (\u03c1=0.0)")
plt.show()
print("With independent features: all FSI values near zero.")
print("No Quadrant II cluster. The IS plot is 'boring' — which is exactly what you want.")

## Comparison: Correlated vs. Independent Features

The contrast between the two IS plots:

| | Breast Cancer (high ρ) | Synthetic (ρ=0.0) |
|---|---|---|
| Quadrant II | Multiple cluster members | Empty (or 1-2 outliers) |
| FSI range | 0 to >1 | Near 0 for all features |
| Reporting strategy | Group important features | Report all individually |

**"Boring is good"**: an IS plot with no Quadrant II features means your explanations are stable and each feature can be reported individually. The Breast Cancer IS plot is alarming — it tells you something is wrong with individual feature-level reporting.

## Summary Decision Flowchart

When you receive DASH outputs, follow this sequence:

1. **Look at the IS plot**
   - Are there features in Quadrant II (top-right)?
   
2. **Quadrant I (Robust Drivers)** → Report individually
   - These features have stable, individually-meaningful attributions
   
3. **Quadrant II (Collinear Cluster)** → Report as group total
   - Identify the cluster using domain knowledge or correlation analysis
   - Report: "The [tumor size] cluster (mean radius, worst perimeter, ...) contributes X% of total importance"
   
4. **Quadrant III (Confirmed Unimportant)** → Omit from report
   - These features are reliably irrelevant across all K models
   
5. **Quadrant IV (Fragile Interactions)** → Investigate
   - These features have unstable attributions despite low mean importance
   - May reflect interaction effects or data artifacts
   
6. **For contested patients** → Use local disagreement map
   - When a patient has features with wide error bars, report cluster-level attributions

## Going Further

DASH produces a `(K × N' × P)` SHAP tensor (`pipe.all_shap_matrices_`) that supports additional analyses not covered here:

- **Robust certification**: which features are in the top-k across *every* K model (not just on average)?
  ```python
  top5_per_model = [set(np.argsort(pipe.all_shap_matrices_[k].mean(0))[-5:]) for k in range(pipe.K)]
  robust_top5 = set.intersection(*top5_per_model)
  print(f"Features in top-5 for ALL {pipe.K} models: {[feature_names[i] for i in robust_top5]}")
  ```

- **Bootstrap confidence intervals** on stability: `stability_bootstrap_ci()` from `dash_shap.evaluation`

- **Extensions**: full framework documented in `docs/API_REFERENCE.md`

See also `notebooks/quickstart.ipynb` cells 10–11 for worked examples of these extensions.